### Implementing baseline models with evasion datasets

This notebook explores the performance of baseline models for spam detection using the **Enron2 dataset** and its **evasion variants** (`charswap`, `homoglyph`, `spacing`). The goal is to evaluate the robustness of machine learning models against adversarial evasion techniques.
#### What we'll do:
1. Train and evaluate baseline models (**Logistic Regression** and **Naive Bayes**) on the original (Enron2) dataset.
2. Assess the performance of these models on evasion datasets when:
   - Models are adversarially trained on evasion datasets.
   - Models are trained only on the original dataset (no adversarial training).
3. Fine-tune the models to optimize their performance.
4. Compare the results to understand the impact of adversarial training.

#### Models Used:
1. **Logistic Regression (LR)**:
   - A linear model that is simple, interpretable, and effective for binary classification tasks like spam detection.
   - It is robust to overfitting when regularization is applied.
2. **Naive Bayes (NB)**:
   - A probabilistic model that assumes feature independence.
   - It is computationally efficient and performs well on text classification tasks, especially when features are sparse (e.g., TF-IDF vectors).

### Importing libraries

In [23]:
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

### Dataset Loading

Loaded the original Enron2 dataset (train, val, test) and its evasion variants (charswap, homoglyph, spacing) from their respective subfolders.
#### Dataset Description:
- **Original Dataset**: Contains spam and ham emails without any modifications.
- **Evasion Datasets**:
  - `charswap`: Emails with character-swapping modifications.
  - `homoglyph`: Emails with homoglyph substitutions (e.g., replacing "o" with "0").
  - `spacing`: Emails with added or removed spaces.

In [24]:
# Define base path to enron2 folder
base_path = "dataset/enron2"

# Load original dataset splits
original_train = pd.read_csv(os.path.join(base_path, "enron2_train.csv"))
original_val = pd.read_csv(os.path.join(base_path, "enron2_val.csv"))
original_test = pd.read_csv(os.path.join(base_path, "enron2_test.csv"))

# Load charswap dataset splits
charswap_train = pd.read_csv(os.path.join(base_path, "charswap/enron2_train_with_charswap.csv"))
charswap_val = pd.read_csv(os.path.join(base_path, "charswap/enron2_val_with_charswap.csv"))
charswap_test = pd.read_csv(os.path.join(base_path, "charswap/enron2_test_with_charswap.csv"))

# Load homoglyph dataset splits
homoglyph_train = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_train_homoglyph.csv"))
homoglyph_val = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_val_homoglyph.csv"))
homoglyph_test = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_test_homoglyph.csv"))

# Load spacing dataset splits
spacing_train = pd.read_csv(os.path.join(base_path, "spacing/enron2_train_spacing.csv"))
spacing_val = pd.read_csv(os.path.join(base_path, "spacing/enron2_val_spacing.csv"))
spacing_test = pd.read_csv(os.path.join(base_path, "spacing/enron2_test_spacing.csv"))

# Check the shape of one dataset to confirm loading
print("Original Train Shape:", original_train.shape)
print(original_train.head())
print("Charswap Train Shape:", charswap_train.shape)
print(charswap_train.head())
print("Homoglyph Train Shape:", homoglyph_train.shape)
print(homoglyph_train.head())
print("Spacing Train Shape:", spacing_train.shape)
print(spacing_train.head())

Original Train Shape: (3727, 2)
                                               email target
0  Subject: membership in the nsf vince : karen m...    ham
1  Subject: holiday gift thank you so much for yo...    ham
2  Subject: re : real options vince , if you take...    ham
3  Subject: men charset = windows - 1252 " > vigo...   spam
4  Subject: vaal medz how t pestilent o save on y...   spam
Charswap Train Shape: (3727, 4)
                                               email target  \
0  Subject: membership in the nsf vince : karen m...    ham   
1  Subject: holiday gift thank you so much for yo...    ham   
2  Subject: re : real options vince , if you take...    ham   
3  Subject: men charset = windows - 1252 " > vigo...   spam   
4  Subject: vaal medz how t pestilent o save on y...   spam   

                                   email_charswapped  was_augmented  
0  Subject: membership in the nsf vince : karen m...          False  
1  Subject: holiday gift thank you so much for yo...     

### Standarization

To ensure consistency across datasets, the following preprocessing steps were applied:
1. **Column Renaming**:
   - For evasion datasets, the modified email content columns (e.g., `email_charswapped`) were renamed to `email_modified`.
   - For the original dataset, the `email` column was renamed to `email_modified`.
2. **Label Encoding**:
   - The `target` column was encoded as:
     - `ham` → 0
     - `spam` → 1

These steps ensure that all datasets have a uniform structure, making them compatible with the machine learning pipeline.

In [25]:
# Function to rename columns and encode labels
def standardize_dataset(df, modified_text_column=None):
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()
    
    # If there's a modified text column, rename it to 'email_modified'
    if modified_text_column:
        df = df.rename(columns={modified_text_column: "email_modified"})
        # Keep only 'email_modified' and 'target' columns, drop others
        df = df[["email_modified", "target"]]
    else:
        # For original dataset, rename 'email' to 'email_modified'
        df = df.rename(columns={"email": "email_modified"})
    
    # Encode the 'target' column: "ham" -> 0, "spam" -> 1
    le = LabelEncoder()
    df["target"] = le.fit_transform(df["target"])
    
    return df

In [26]:
# Standardize all datasets
# Original dataset (no modified text column)
original_train = standardize_dataset(original_train)
original_val = standardize_dataset(original_val)
original_test = standardize_dataset(original_test)

# Charswap dataset
charswap_train = standardize_dataset(charswap_train, "email_charswapped")
charswap_val = standardize_dataset(charswap_val, "email_charswapped")
charswap_test = standardize_dataset(charswap_test, "email_charswapped")

# Homoglyph dataset
homoglyph_train = standardize_dataset(homoglyph_train, "email_homoglyph")
homoglyph_val = standardize_dataset(homoglyph_val, "email_homoglyph")
homoglyph_test = standardize_dataset(homoglyph_test, "email_homoglyph")

# Spacing dataset
spacing_train = standardize_dataset(spacing_train, "email_spaced")
spacing_val = standardize_dataset(spacing_val, "email_spaced")
spacing_test = standardize_dataset(spacing_test, "email_spaced")

In [27]:
# Check the standardized datasets
print("Standardized Original Train Shape:", original_train.shape)
print(original_train.head())
print("Standardized Charswap Train Shape:", charswap_train.shape)
print(charswap_train.head())
print("Standardized Homoglyph Train Shape:", homoglyph_train.shape)
print(homoglyph_train.head())
print("Standardized Spacing Train Shape:", spacing_train.shape)
print(spacing_train.head())

Standardized Original Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject: holiday gift thank you so much for yo...       0
2  Subject: re : real options vince , if you take...       0
3  Subject: men charset = windows - 1252 " > vigo...       1
4  Subject: vaal medz how t pestilent o save on y...       1
Standardized Charswap Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject: holiday gift thank you so much for yo...       0
2  Subject: re : real options vince , if you take...       0
3  ['Subject: men charset = windows - 1252 "> vig...       1
4  ['Subject: aval medz how t pestilent o svae on...       1
Standardized Homoglyph Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject

### Preprocessing and Vectorization

The email content was converted into numerical features using **TF-IDF vectorization**. This technique transforms text data into a sparse matrix of numerical values, where each value represents the importance of a word in a document relative to the entire dataset.

#### Why TF-IDF?
- It captures the importance of words while reducing the impact of commonly occurring words (e.g., "the", "and").
- It is well-suited for text classification tasks, especially when combined with models like Logistic Regression and Naive Bayes.

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Function to preprocess and vectorize text data
def preprocess_and_vectorize(train_data, val_data, test_data, text_column="email_modified"):
    vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
    
    # Fit the vectorizer on the training data and transform all splits
    X_train = vectorizer.fit_transform(train_data[text_column])
    X_val = vectorizer.transform(val_data[text_column])
    X_test = vectorizer.transform(test_data[text_column])
    
    # Extract labels
    y_train = train_data["target"]
    y_val = val_data["target"]
    y_test = test_data["target"]
    
    return X_train, X_val, X_test, y_train, y_val, y_test, vectorizer

In [29]:
# Apply preprocessing to each dataset
datasets = {
    "original": (original_train, original_val, original_test),
    "charswap": (charswap_train, charswap_val, charswap_test),
    "homoglyph": (homoglyph_train, homoglyph_val, homoglyph_test),
    "spacing": (spacing_train, spacing_val, spacing_test)
}

# Dictionary to store vectorized data
vectorized_data = {}
for name, (train, val, test) in datasets.items():
    X_train, X_val, X_test, y_train, y_val, y_test, vectorizer = preprocess_and_vectorize(train, val, test)
    vectorized_data[name] = (X_train, X_val, X_test, y_train, y_val, y_test)
    print(f"{name} - X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}")

original - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
charswap - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
homoglyph - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
spacing - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)


### Model (Adversarial) Training and Evaluation

The models were trained on the original dataset as well as adversarially on the evasion datasets. This allows us to evaluate their robustness against different types of evasion techniques.

#### Metrics:
The following metrics were calculated for both validation and test sets:
- **Accuracy**: Overall correctness of the model.
- **Precision**: Proportion of correctly identified spam emails out of all predicted spam emails.
- **Recall**: Proportion of correctly identified spam emails out of all actual spam emails.
- **F1-Score**: Harmonic mean of precision and recall, balancing false positives and false negatives.

#### Models:
1. **Logistic Regression**:
   - Trained with regularization to prevent overfitting.
   - Fine-tuned using `GridSearchCV` to optimize the regularization parameter (`C`).
2. **Naive Bayes**:
   - Trained with Laplace smoothing (`alpha` parameter).
   - Fine-tuned using `GridSearchCV` to find the optimal `alpha` value.

#### Results:
The models were evaluated on both the original and evasion datasets to compare their performance under adversarial conditions.

In [30]:
from sklearn.metrics import classification_report

def train_and_evaluate(classifier, classifier_name, X_train, y_train, X_test, y_test, dataset_name):
    # Train the classifier
    classifier.fit(X_train, y_train)

    # Predict on the test set
    y_pred = classifier.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    f1 = f1_score(y_test, y_pred, average='binary')

    # Print the results
    print(f"\nResults for {dataset_name} dataset ({classifier_name}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}\n")

    print("Detailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

    return classifier

In [31]:
# Naive Bayes and Logistic Regression implementation
from sklearn.naive_bayes import MultinomialNB

for dataset_name, (X_train, X_val, X_test, y_train, y_val, y_test) in vectorized_data.items():
    print(f"\nProcessing {dataset_name} dataset...")

    # Train and evaluate Naive Bayes
    nb_classifier = MultinomialNB()
    train_and_evaluate(nb_classifier, "Naïve Bayes", X_train, y_train, X_test, y_test, dataset_name)

    # Train and evaluate Logistic Regression
    lr_classifier = LogisticRegression(max_iter=1000)
    train_and_evaluate(lr_classifier, "Logistic Regression", X_train, y_train, X_test, y_test, dataset_name)


Processing original dataset...

Results for original dataset (Naïve Bayes):
Accuracy: 0.9820
Precision: 0.9929
Recall: 0.9365
F1-Score: 0.9639

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       866
        spam       0.99      0.94      0.96       299

    accuracy                           0.98      1165
   macro avg       0.99      0.97      0.98      1165
weighted avg       0.98      0.98      0.98      1165


Results for original dataset (Logistic Regression):
Accuracy: 0.9845
Precision: 0.9930
Recall: 0.9465
F1-Score: 0.9692

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       866
        spam       0.99      0.95      0.97       299

    accuracy                           0.98      1165
   macro avg       0.99      0.97      0.98      1165
weighted avg       0.98      0.98      0.98      1165


Processing char

In [35]:
from sklearn.model_selection import GridSearchCV
def fine_tune_and_evaluate(classifier, param_grid, X_train, X_val, y_train, y_val, X_test, y_test, classifier_name, dataset_name):
    # Perform grid search for hyperparameter tuning
    grid_search = GridSearchCV(classifier, param_grid, cv=3, scoring="f1", n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Get the best model
    best_model = grid_search.best_estimator_
    print(f"\nBest parameters for {dataset_name} dataset ({classifier_name}): {grid_search.best_params_}")

    # Evaluate on validation set
    y_val_pred = best_model.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred)
    print(f"Validation F1-Score for {dataset_name} dataset ({classifier_name}): {val_f1:.4f}")

    # Evaluate on test set
    y_test_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred, average='binary')
    recall = recall_score(y_test, y_test_pred, average='binary')
    f1 = f1_score(y_test, y_test_pred, average='binary')

    # Print the results
    print(f"\nResults for {dataset_name} dataset ({classifier_name}) after fine-tuning:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}\n")

    print("Detailed Classification Report:")
    print(classification_report(y_test, y_test_pred, target_names=['ham', 'spam']))

    return best_model

In [36]:
# Define parameter grids for fine-tuning
lr_param_grid = {"C": [0.01, 0.1, 1, 10, 100]}
nb_param_grid = {"alpha": [0.01, 0.1, 0.5, 1, 5, 10]}

for dataset_name, (X_train, X_val, X_test, y_train, y_val, y_test) in vectorized_data.items():
    print(f"\nProcessing {dataset_name} dataset...")

    # Fine-tune and evaluate Logistic Regression
    lr_classifier = LogisticRegression(max_iter=1000)
    fine_tune_and_evaluate(lr_classifier, lr_param_grid, X_train, X_val, y_train, y_val, X_test, y_test, "Logistic Regression", dataset_name)

    # Fine-tune and evaluate Naive Bayes
    nb_classifier = MultinomialNB()
    fine_tune_and_evaluate(nb_classifier, nb_param_grid, X_train, X_val, y_train, y_val, X_test, y_test, "Naïve Bayes", dataset_name)


Processing original dataset...

Best parameters for original dataset (Logistic Regression): {'C': 100}
Validation F1-Score for original dataset (Logistic Regression): 0.9895

Results for original dataset (Logistic Regression) after fine-tuning:
Accuracy: 0.9948
Precision: 0.9900
Recall: 0.9900
F1-Score: 0.9900

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00       866
        spam       0.99      0.99      0.99       299

    accuracy                           0.99      1165
   macro avg       0.99      0.99      0.99      1165
weighted avg       0.99      0.99      0.99      1165


Best parameters for original dataset (Naïve Bayes): {'alpha': 0.1}
Validation F1-Score for original dataset (Naïve Bayes): 0.9832

Results for original dataset (Naïve Bayes) after fine-tuning:
Accuracy: 0.9871
Precision: 0.9897
Recall: 0.9599
F1-Score: 0.9745

Detailed Classification Report:
              precision    recall

### Working without adversarial training

To simulate a real-world scenario where the models are not adversarially trained, they were trained only on the original dataset and evaluated on the evasion datasets.

This experiment highlights the importance of adversarial training by showing how models trained only on clean data perform poorly on adversarially modified data.

The performance metrics (accuracy, precision, recall, F1-score) were calculated for each evasion dataset. These results were compared with those from adversarially trained models to demonstrate the impact of adversarial training.

In [37]:
# Train models on the original dataset and evaluate on evasion datasets
models_original_train = {}

# Train on original dataset
X_train_original, X_val_original, X_test_original, y_train_original, y_val_original, y_test_original = vectorized_data["original"]

# Train Logistic Regression and Naive Bayes on original training data
lr_model_original = LogisticRegression(max_iter=1000)
lr_model_original.fit(X_train_original, y_train_original)

nb_model_original = MultinomialNB()
nb_model_original.fit(X_train_original, y_train_original)

# Store the models
models_original_train["Logistic Regression"] = lr_model_original
models_original_train["Naive Bayes"] = nb_model_original

# Evaluate on evasion datasets
for evasion_name in ["charswap", "homoglyph", "spacing"]:
    print(f"\nEvaluating models trained on original dataset on {evasion_name} dataset...")
    
    # Get the evasion dataset splits
    X_val_evasion, X_test_evasion, y_val_evasion, y_test_evasion = (
        vectorized_data[evasion_name][1],  # Validation features
        vectorized_data[evasion_name][2],  # Test features
        vectorized_data[evasion_name][4],  # Validation labels
        vectorized_data[evasion_name][5],  # Test labels
    )
    
    # Evaluate Logistic Regression
    y_test_pred_lr = lr_model_original.predict(X_test_evasion)
    accuracy_lr = accuracy_score(y_test_evasion, y_test_pred_lr)
    precision_lr = precision_score(y_test_evasion, y_test_pred_lr)
    recall_lr = recall_score(y_test_evasion, y_test_pred_lr)
    f1_lr = f1_score(y_test_evasion, y_test_pred_lr)
    print(f"\nResults for {evasion_name} dataset (Logistic Regression):")
    print(f"Accuracy: {accuracy_lr:.4f}")
    print(f"Precision: {precision_lr:.4f}")
    print(f"Recall: {recall_lr:.4f}")
    print(f"F1-Score: {f1_lr:.4f}\n")
    print("Detailed Classification Report:")
    print(classification_report(y_test_evasion, y_test_pred_lr, target_names=['ham', 'spam']))
    
    # Evaluate Naive Bayes
    y_test_pred_nb = nb_model_original.predict(X_test_evasion)
    accuracy_nb = accuracy_score(y_test_evasion, y_test_pred_nb)
    precision_nb = precision_score(y_test_evasion, y_test_pred_nb)
    recall_nb = recall_score(y_test_evasion, y_test_pred_nb)
    f1_nb = f1_score(y_test_evasion, y_test_pred_nb)
    print(f"\nResults for {evasion_name} dataset (Naïve Bayes):")
    print(f"Accuracy: {accuracy_nb:.4f}")
    print(f"Precision: {precision_nb:.4f}")
    print(f"Recall: {recall_nb:.4f}")
    print(f"F1-Score: {f1_nb:.4f}\n")
    print("Detailed Classification Report:")
    print(classification_report(y_test_evasion, y_test_pred_nb, target_names=['ham', 'spam']))


Evaluating models trained on original dataset on charswap dataset...

Results for charswap dataset (Logistic Regression):
Accuracy: 0.7536
Precision: 0.7000
Recall: 0.0702
F1-Score: 0.1277

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.76      0.99      0.86       866
        spam       0.70      0.07      0.13       299

    accuracy                           0.75      1165
   macro avg       0.73      0.53      0.49      1165
weighted avg       0.74      0.75      0.67      1165


Results for charswap dataset (Naïve Bayes):
Accuracy: 0.6953
Precision: 0.4222
Recall: 0.5084
F1-Score: 0.4613

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.82      0.76      0.79       866
        spam       0.42      0.51      0.46       299

    accuracy                           0.70      1165
   macro avg       0.62      0.63      0.62      1165
weighted avg       0.72      0.70

### About the results 
The models LR and NB performed very good on its evaluations regarding the original Enron2 dataset, as well as with the variations of the dataset (charswap, homoglyph, spacing) as they were adversarially trained and the evaluation was performed individually. 

Logistic Regression outperformed Naive Bayes across all evasion datasets, achieving higher F1-scores and better overall performance. Naive Bayes, while still effective, struggled more with the adversarial modifications due to its independence assumption, which limits its ability to capture complex patterns introduced by evasion techniques.

When working without adversarial training, both models experienced a significant drop in performance when evaluated on the evasion datasets. This highlights the vulnerability of models trained only on clean data to adversarial modifications, as they fail to generalize to adversarially altered inputs.
